In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/Jack1021ohoh/Pitch_Sequence_Optimization.git
%cd Pitch_Sequence_Optimization

Cloning into 'Pitch_Sequence_Optimization'...
remote: Enumerating objects: 262, done.
remote: Counting objects: 100% (262/262), done.
remote: Compressing objects: 100% (181/181), done.
remote: Total 262 (delta 155), reused 185 (delta 78), pack-reused 0 (from 0)
Receiving objects: 100% (262/262), 127.82 KiB | 1.77 MiB/s, done.
Resolving deltas: 100% (155/155), done.
/content/Pitch_Sequence_Optimization


In [3]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.1/426.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.2 MB/s eta 0:00:00


In [4]:
import os
os.environ['DATA_DIR']     = '/content/drive/MyDrive/pitch_sequence/data'
os.environ['BASELINE_DIR'] = '/content/drive/MyDrive/pitch_sequence/baselines'

In [5]:
import shutil

DRIVE_DATA = '/content/drive/MyDrive/pitch_sequence/data'
LOCAL_DATA = '/content/pitch_sequence/data'

os.makedirs(f'{LOCAL_DATA}/processed', exist_ok=True)
os.makedirs(f'{LOCAL_DATA}/artifacts', exist_ok=True)

for f in ['pitches_train.parquet', 'pitches_val.parquet', 'pitches_test.parquet']:
    shutil.copy(f'{DRIVE_DATA}/processed/{f}', f'{LOCAL_DATA}/processed/{f}')
    print(f'Copied {f}')

shutil.copy(f'{DRIVE_DATA}/artifacts/class_weights.json', f'{LOCAL_DATA}/artifacts/class_weights.json')
print('Copied class_weights.json')

Copied pitches_train.parquet
Copied pitches_val.parquet
Copied pitches_test.parquet
Copied class_weights.json


In [6]:
%run baselines/train_lightgbm.py

Loading outcome data...

Training outcome model  (train=2,164,776  val=726,388)
Training until validation scores don't improve for 20 rounds
[50]	valid_0's multi_logloss: 1.25935
[100]	valid_0's multi_logloss: 1.23155
[150]	valid_0's multi_logloss: 1.23226
Early stopping, best iteration is:
[135]	valid_0's multi_logloss: 1.23073
Saved -> /content/drive/MyDrive/pitch_sequence/baselines/lgbm_outcome.txt

Loading hit location data...

Training hit location model  (train=371,673  val=126,042)
Training until validation scores don't improve for 20 rounds
[50]	valid_0's multi_logloss: 2.15631
[100]	valid_0's multi_logloss: 2.15286
Early stopping, best iteration is:
[81]	valid_0's multi_logloss: 2.15245
Saved -> /content/drive/MyDrive/pitch_sequence/baselines/lgbm_location.txt


In [7]:
%run baselines/evaluate_lightgbm.py

Loading models from /content/drive/MyDrive/pitch_sequence/baselines...
Loading test data...

── Pitch Outcome (Test 2025) ───────────────────
  Top-4 recall    : 0.9581
  Top-1 accuracy  : 0.5757
  Log-loss        : 1.1137
  Brier score     : 0.5573
  ECE             : 0.0461

  Per-class Top-4 recall:
    Ball            : 0.9738
    Strike          : 0.9521
    Single          : 0.9886
    Double          : 0.4856
    Triple          : 0.0091
    Home Run        : 0.2463
    Strikeout       : 1.0000
    Walk            : 0.9898
    Hit by Pitch    : 0.9489
    Field Out       : 0.9970

── Hit Location (Test 2025) ────────────────────
  Top-4 recall    : 0.6417
  Top-1 accuracy  : 0.1845
  Log-loss        : 2.1330
  Brier score     : 0.8687
  ECE             : 0.0167

  Per-class Top-4 recall:
    Pitcher         : 0.5976
    Catcher         : 0.1663
    First Base      : 0.6199
    Second Base     : 0.6386
    Third Base      : 0.6234
    Shortstop       : 0.6045
    Left Field      